# Метрики регрессии

**Задача регрессии** заключается в предсказании непрерывной целевой переменной. При этом важно, чтобы построенная модель предсказывала значения таким образом, чтобы отклонения реального значения и прогноза модели были минимальными.

<img src='https://miro.medium.com/max/864/0*OVG0UBquPwSiwiUi' width=600>

## Получение данных

Посмотрим на метрики на примере набора данных с характеристиками домов в Бостоне, где целевая переменная - стоимость дома.

In [1]:
import pandas as pd
import numpy as np

In [4]:
from sklearn.datasets import load_diabetes

data = load_diabetes()
data.keys()

dict_keys(['data', 'target', 'frame', 'DESCR', 'feature_names', 'data_filename', 'target_filename', 'data_module'])

In [5]:
print(data.DESCR)

.. _diabetes_dataset:

Diabetes dataset
----------------

Ten baseline variables, age, sex, body mass index, average blood
pressure, and six blood serum measurements were obtained for each of n =
442 diabetes patients, as well as the response of interest, a
quantitative measure of disease progression one year after baseline.

**Data Set Characteristics:**

:Number of Instances: 442

:Number of Attributes: First 10 columns are numeric predictive values

:Target: Column 11 is a quantitative measure of disease progression one year after baseline

:Attribute Information:
    - age     age in years
    - sex
    - bmi     body mass index
    - bp      average blood pressure
    - s1      tc, total serum cholesterol
    - s2      ldl, low-density lipoproteins
    - s3      hdl, high-density lipoproteins
    - s4      tch, total cholesterol / HDL
    - s5      ltg, possibly log of serum triglycerides level
    - s6      glu, blood sugar level

Note: Each of these 10 feature variables have bee

In [6]:
X = data.data
features = data.feature_names
y = data.target

Из признаков и целевой переменной сформируем датафрейм, где в качестве названий колонок возьмем названия признаков.

In [7]:
df = pd.DataFrame(X, columns=features)
df['target'] = y

df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


## Обучение модели

Возьмем линейную модель из `sklearn'a` и обучим на выборке.



In [8]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X, y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


Теперь сделаем предсказания на всей выборке, чтобы понять, насколько хорошо модель научилась предсказывать стоимость дома.

In [9]:
pred = lr.predict(X)

## Метрики

Выведем истинные значения целевой переменной и предсказанные моделью значения стоимости дома.

In [10]:
pred_df = pd.DataFrame({
    'y truth': y,
    'prediction': pred
})

pred_df

,y truth,prediction
0,151.0,206.116677
1,75.0,68.071033
2,141.0,176.882790
3,206.0,166.914458
4,135.0,128.462258
...,...,...
437,178.0,193.017565
438,104.0,101.055814
439,132.0,121.226681
440,220.0,211.858945


В первой строке модель должна была предсказать 151.0, а предсказала 206.11667, во второй строке истина 75.0, а модель сказала 68.071033. Так мы можем пройтись по каждому объекту и посмотреть, насколько близким оказался прогноз.

In [12]:
pred_df['difference'] = pred_df['prediction'] - pred_df['y truth']
pred_df

,y truth,prediction,difference
0,151.0,206.116677,55.116677
1,75.0,68.071033,-6.928967
2,141.0,176.882790,35.882790
3,206.0,166.914458,-39.085542
4,135.0,128.462258,-6.537742
...,...,...,...
437,178.0,193.017565,15.017565
438,104.0,101.055814,-2.944186
439,132.0,121.226681,-10.773319
440,220.0,211.858945,-8.141055


В столбце `difference` посчитана разница между предсказанием и истинным значением. Чтобы оценить это разницу в среденем по всей выборке просуммируем отклонения и поделим на количество строк.

In [13]:
pred_df['difference'].sum() / pred_df.shape[0]

np.float64(2.829310441035693e-15)

Таким образом, мы получили число приближенное к нулю, что как будто говорит нам, что в среднем наши предсказания близки к истине, но выше мы наблюдали, что это не так.
Усредненные отклонения в среднем дали такую маленькую ошибку из-за того, что мы просуммировали как положительную (в случае предсказаний больше истины), так и отрицательную разницы (в случае предсказаний меньше истины).А когда мы суммируем положительные и отрицательные значения, они друг друга нивелируют, и по итогу получается небольшое значение.

Получается, что нам нужно избавиться от знака минуса в наших предсказаниях.

### MAE

Для этого мы можем брать абсолютные значения, то есть брать модуль числа:

In [14]:
pred_df['abs'] = abs(pred_df['prediction'] - pred_df['y truth'])
pred_df

,y truth,prediction,difference,abs
0,151.0,206.116677,55.116677,55.116677
1,75.0,68.071033,-6.928967,6.928967
2,141.0,176.882790,35.882790,35.882790
3,206.0,166.914458,-39.085542,39.085542
4,135.0,128.462258,-6.537742,6.537742
...,...,...,...,...
437,178.0,193.017565,15.017565,15.017565
438,104.0,101.055814,-2.944186,2.944186
439,132.0,121.226681,-10.773319,10.773319
440,220.0,211.858945,-8.141055,8.141055


Теперь не будет такого, что отрицательные и положительные отклонения складываются в ноль.   
Найдем среднее отклонение по модулю для всей выборки:

In [15]:
pred_df['abs'].mean()

np.float64(43.27745202531506)

Получается мы в среднем ошибаемся на 3.27 **единиц** по всей выборке. Это мы подсчитали метрику под названием **MAE (mean absolute error, средне-абсолютная ошибка)**.

$$MAE = \frac{1}{n}\sum_i^n{|y - y_{pred}|}$$

Она реализована в библиотеки `sklearn` в модуле `metrics`:


In [16]:
from sklearn.metrics import mean_absolute_error

In [17]:
mean_absolute_error(pred_df['y truth'], pred_df['prediction'])

43.27745202531506

Получается такое же значения метрики, как мы получали при ручном подсчете.   
**Чем метрика MAE ближе к нулю, тем лучше, в этом случае мы попадаем во все истинные значения**.

### MSE

Как ещё мы могли бы избавиться от знака минуса в отклонениях?  
Мы могли бы возводить отклонения в квадрат, а далее так же искать среднее по выборке:

In [28]:
pred_df['squared'] = (pred_df['prediction'] - pred_df['y truth']) ** 2
pred_df

,y truth,prediction,difference,abs,squared
0,151.0,206.116677,55.116677,55.116677,3037.848111
1,75.0,68.071033,-6.928967,6.928967,48.010584
2,141.0,176.882790,35.882790,35.882790,1287.574643
3,206.0,166.914458,-39.085542,39.085542,1527.679560
4,135.0,128.462258,-6.537742,6.537742,42.742066
...,...,...,...,...,...
437,178.0,193.017565,15.017565,15.017565,225.527273
438,104.0,101.055814,-2.944186,2.944186,8.668233
439,132.0,121.226681,-10.773319,10.773319,116.064397
440,220.0,211.858945,-8.141055,8.141055,66.276774


Снова не будет отрицательных отклонений, которые ведут к очень маленькому усреднению.   
Найдем среднее отклонение в квадрате для всей выборки:

In [29]:
pred_df['squared'].mean()

np.float64(2859.6963475867506)

Получается мы в среднем ошибаемся на 2859.69 **квадратичных единиц** по всей выборке. Эта метрика называется **MSE (mean squared error, средне-квадратичная ошибка)**.

$$MSE = \frac{1}{n}\sum_i^n{(y - y_{pred})^2}$$

Она тоже реализована в библиотеки `sklearn` в модуле `metrics`:


In [32]:
from sklearn.metrics import mean_squared_error

In [34]:
mean_squared_error(pred_df['y truth'], pred_df['prediction'])

2859.6963475867506

И снова наши значения совпали.   
**Чем метрика MSE ближе к нулю, тем лучше.**

MSE используется под капотом моделей машиного обучения в качестве **функция потерь**.

Однако когда мы хотим понять, насколько модель ошибается, метрика MSE может вносить неясность из-за большого штраф за ошибку (возведение в квадрат) и сложности интерпретации. 

### RMSE

Выше отклонения мы считали в квадрате, поэтому если наша стоимость изменяется в рублях, то MSE дает цифру рубли в квадрате, что усложняет интерпретацию результата, это легко исправить взятием корня квадратного из значения MSE - а это и есть новая метрика **RMSE**.

Для подсчета этой метрики можно усреднить квадратичные отклонения (то есть получить MSE), а затем взять корень квадратный:

In [35]:
np.sqrt(pred_df['squared'].mean())

np.float64(53.476128764026576)

Теперь очевидно, что в среднем ошибаемся на 53.47 **единиц** по всей выборке, либо в большую, либо в меньшую сторону. Эта метрика **RMSE (root mean squared error)**.

$$RMSE = \sqrt{\frac{1}{n}\sum_i^n{(y - y_{pred})^2}}$$

In [36]:
np.sqrt(mean_squared_error(pred_df['y truth'], pred_df['prediction']))

np.float64(53.476128764026576)

Значения идентичны.
И так же, как предыдущие метрики, **чем метрика RMSE ближе к нулю, тем лучш**е.

### $R^2$

Три рассмотренные ранее метрики имеют "идеальное" значение 0, однако у них нет порогового значения выше которого мы однозначно могли бы сказать, что модель плоха, и зависит это от исходных данных.

К примеру, если наша целевая переменная изменяется от 0 до 10:

In [40]:
y_true = [0, 3, 6, 2, 9]

Наша модель может предсказать:

In [38]:
y_pred = [1, 5, 9, 6, 14]

Соединим предсказания и истину вместе и найдем разницу между ними:

In [41]:
df = pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred
})

df['difference'] = df['y_pred'] - df['y_true']
df

,y_true,y_pred,difference
0,0,1,1
1,3,5,2
2,6,9,3
3,2,6,4
4,9,14,5


И мы получим следующее значение средне-квадратичной ошибки:

In [47]:
mean_squared_error(y_true, y_pred)

11.0

А если наша целевая переменная изменяется от 0 до 10000:

In [48]:
y_true = [0, 300, 6000, 2000, 9900]

Наша модель может предсказать:

In [49]:
y_pred = [1, 302, 6003, 2004, 9905]

Соединим предсказания и истину вместе и найдем разницу между ними:

In [50]:
df = pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred
})

df['difference'] = df['y_pred'] - df['y_true']
df

,y_true,y_pred,difference
0,0,1,1
1,300,302,2
2,6000,6003,3
3,2000,2004,4
4,9900,9905,5


И мы получим следующее значение средне-квадратичной ошибки:

In [51]:
mean_squared_error(y_true, y_pred)

11.0

MSE получились одинаковыми, потому что одинаковые разницы в предсказаниях, но вторая модель очевидно делает меньше ошибок, чем первая, значит она более качественная, хотя значения метрики MSE абсолютно одинаковые.

MSE не позволяет сама по себе сделать вывод о том, как хорошо модель решает задачу. Вот как в нашем примере, если целевая переменная принимает значения от 0 до 10, а MSE равняется 11, это плохой показатель, а когда целевая переменная варьируется от 0 до 10000, такое же значение уже является очень хорошим.  
Для избавления от такой неясности и был введен **коэффициент детерминации**, который по сути является нормированной среднеквадратичной ошибкой и принимает значения от 0 до 1.


$$R^2 = 1 - \frac{\frac{1}{n}\sum^{n}_{i}{(y - y_{pred})^2}}{\frac{1}{n}\sum^{n}_{i}{(y - \bar{y})^2}}$$

где $\bar{y}=\frac{1}{l}\sum^{n}_{i}y_{i}$ - среднее значение целевой переменной.

Коэффициент детерминации характеризует, какую долю дисперсии ответов объясняет модель. Если $R^{2}=1$, то модель идеально описывает данные, если же $R^{2}$ близко к нулю, то предсказания сопоставимы по качеству с константной моделью.  
То есть, можем относиться к метрике $R^{2}$ как к точности работы модели для задачи регрессии.

Для подсчета этой метрики **в числителе используется MSE** нашей модели, а в **знаменателе дисперсия целевой переменной** (хотя к этому значению можно относиться, как к MSE константной модели, которая везде отвечает среднее по выборке):

In [52]:
pred_df['constant'] = pred_df['y truth'].mean()
pred_df

,y truth,prediction,difference,abs,squared,constant
0,151.0,206.116677,55.116677,55.116677,3037.848111,152.133484
1,75.0,68.071033,-6.928967,6.928967,48.010584,152.133484
2,141.0,176.882790,35.882790,35.882790,1287.574643,152.133484
3,206.0,166.914458,-39.085542,39.085542,1527.679560,152.133484
4,135.0,128.462258,-6.537742,6.537742,42.742066,152.133484
...,...,...,...,...,...,...
437,178.0,193.017565,15.017565,15.017565,225.527273,152.133484
438,104.0,101.055814,-2.944186,2.944186,8.668233,152.133484
439,132.0,121.226681,-10.773319,10.773319,116.064397,152.133484
440,220.0,211.858945,-8.141055,8.141055,66.276774,152.133484


Получаем, что MSE константной модели, которая везде отвечает среднее по выборке равно

In [53]:
mse_const = mean_squared_error(pred_df['y truth'], pred_df['constant'])
mse_const

5929.884896910383

А MSE нашей модели получается следующим

In [54]:
mse_model = mean_squared_error(pred_df['y truth'], pred_df['prediction'])
mse_model

2859.6963475867506

Исходя из получившихся значений видно, что наша модель уловила больше вариативности данных, нежели модель со средним по выборке.

Теперь мы можем посчитать коэффициент детерминации: поделим MSE нашей модели на дисперсию целевой переменной, а затем это значение вычтем из 1:

In [57]:
1 - mse_model / mse_const

0.5177484222203498

$R^2$ реализована в библиотеке `sklearn`:

In [58]:
from sklearn.metrics import r2_score

r2_score(pred_df['y truth'], pred_df['prediction'])

0.5177484222203498

Получается $R^2$ нашей модели далеко от 1, что может говорит о хорошем качестве модели.

#### Отрицательный $R^2$

Иногда случается так, что модель получается на столько плохой, что её ответы никак не соотносятся с истинными, в этом случае коэффициент детерминации может быть отрицательным.


In [59]:
y_true = [0, 1, 2, 3, 4]
y_pred = [10, -20, -30, 40, -50]


df = pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred
})


df['difference'] = df['y_pred'] - df['y_true']
df

,y_true,y_pred,difference
0,0,10,10
1,1,-20,-21
2,2,-30,-32
3,3,40,37
4,4,-50,-54


In [60]:
variance = ((df['y_true'] - df['y_true'].mean()) ** 2).mean()
print(f'Дисперсия выборки {variance}')

Дисперсия выборки 2.0


In [61]:
mse = ((df['y_true'] - df['y_pred']) ** 2).mean()
print(f'MSE {mse}')

MSE 1170.0


Полученная MSE гораздо больше дисперсии выборки, из-за этого и получается большое число при делении MSE на MSE константной модели, которая везде отвечает среднее по выборке.

In [62]:
mse / variance

np.float64(585.0)

Вычитая полученное число из 1 и получается отрицательное значение:

In [63]:
1 - mse / variance

np.float64(-584.0)

Метрика $R^2$ через sklearn.

In [64]:
r2_score(y_true, y_pred)

-584.0

Подобное значение может сигнализировать о том, что модель либо переобучилась, либо при предсказывании объектов была использована неправильная предобработка данных.

## Резюме

<table>

<tr>
<td>
Метрика
</td>

<td>
Формула
</td>

<td>
Диапазоны значений
</td>

<td>
Идеальное значение
</td>
</tr>

<tr>
<td>
MAE (mean absolute error, средне-абсолютная ошибка)
</td>

<td>
$$MAE = \frac{1}{n}\sum_i^n{|y - y_{pred}|}$$

</td>

<td>
[0, +$\infty$)
</td>

<td>
0
</td>
</tr>

<tr>
<td>
MSE (mean squared error, средне-квадратичная ошибка)

</td>

<td>
$$MSE = \frac{1}{n}\sum_i^n{(y - y_{pred})^2}$$

</td>

<td>
[0, +$\infty$)
</td>

<td>
0
</td>
</tr>

<tr>
<td>
RMSE (root mean squared error).

</td>

<td>
$$RMSE = \sqrt{\frac{1}{n}\sum_i^n{(y - y_{pred})^2}}$$

</td>

<td>
[0, +$\infty$)
</td>

<td>
0
</td>
</tr>

<tr>
<td>
Коэффициент детерминации $R^{2}$

</td>

<td>
$$R^2 = 1 - \frac{\frac{1}{n}\sum^{n}_{i}{(y - y_{pred})^2}}{\frac{1}{n}\sum^{n}_{i}{(y - \bar{y})^2}}$$

</td>

<td>
(-$\infty$, 1]

</td>

<td>
1
</td>
</tr>
</table>

Метрики используются для того, чтобы получить цифру оценки работы обученной модели, чтобы уже заранее знать, чего ожидать от модели, когда ей будем давать реальные данные, а так же метрики мы можем использовать для обнаружения переобучения/недообучения